# Does the earliest-hook result survive a scaling check?

This is an exploratory response to an observed failure. At fixed L01, each feature scale is floored at the median retained-feature SD, computed inside every training split. No test-derived clipping or row deletion occurs. This does not replace the original all-hook comparison.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.report_bundle import read_summary, verify_bundle
from src.utils import focus_table, display_details
REPORTS = Path(os.environ.get("SAERT_REPORTS_DIR", ROOT/"reports"))
verify_bundle(REPORTS)
def load(section, name): return read_summary(section, name, REPORTS)
def figure(section, name): display(Image(filename=str(REPORTS/section/(name+".png"))))


In [2]:
raw = load("scaling_sensitivity","comparison")
view = pd.DataFrame({"Corpus":raw.corpus.map({"provo":"Provo","natural_stories":"Natural Stories"}),
 "State":raw.state.map({"prefix":"Before word","post":"After word"}),"Gain with floor (pp)":raw.floor_gain_pp,
 "95% interval":[f"[{a:+.2f}, {b:+.2f}]" for a,b in zip(raw.floor_ci_low_pp,raw.floor_ci_high_pp)]})
display(focus_table(view,important=["Gain with floor (pp)"],signed_columns=["Gain with floor (pp)"],formats={"Gain with floor (pp)":"{:+.2f}"}))
errors=view[["Corpus","State"]].copy()
errors["Original max error"]=raw.original_max_abs_error
errors["Floor max error"]=raw.floor_max_abs_error
display(focus_table(errors,important=["Floor max error"],formats={"Original max error":"{:.2f}","Floor max error":"{:.2f}"},caption="Worst absolute prediction error in log milliseconds, not average error."))

Corpus,State,Gain with floor (pp),95% interval
Provo,Before word,+3.41,"[+2.53, +4.33]"
Provo,After word,+6.53,"[+5.28, +7.82]"
Natural Stories,Before word,+0.62,"[+0.20, +0.95]"
Natural Stories,After word,+2.13,"[+1.47, +2.81]"


Corpus,State,Original max error,Floor max error
Provo,Before word,0.40,0.40
Provo,After word,0.41,0.40
Natural Stories,Before word,69.47,1.07
Natural Stories,After word,111.88,1.10


Provo’s after-word gain is nearly unchanged (+6.58 to +6.53 pp). The extreme Natural Stories first-hook errors disappear. This does not prove optimal scaling or establish what an all-hook comparison would find. Next: lexical identity/position controls and a declared all-hook sensitivity.

In [3]:
display_details("Complete comparison, including original failures",raw)

corpus,state,hook,original_gain_pp,floor_gain_pp,floor_ci_low_pp,floor_ci_high_pp,floor_minus_original_pp,original_max_abs_error,floor_max_abs_error,texts_improved,n_texts
provo,prefix,L01,2.869,3.412,2.533,4.333,0.543,0.396,0.404,44,55
provo,post,L01,6.576,6.531,5.277,7.819,-0.046,0.406,0.401,48,55
natural_stories,prefix,L01,-3521.033,0.617,0.200,0.949,3521.649,69.470,1.075,9,10
natural_stories,post,L01,-9131.484,2.128,1.469,2.814,9133.613,111.878,1.098,10,10
